In [5]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import randint, uniform
import warnings

warnings.filterwarnings("ignore")

# --- 1. Data Loading and Preprocessing ---
print("Step 1: Loading and Preprocessing Data...")

feature_path = r"C:\Users\samee\Downloads\EEG_Features"  # folder with .npz files

all_features = []
all_labels = []

# Load all .npz files in the folder
for file in os.listdir(feature_path):
    if file.endswith(".npz"):
        filepath = os.path.join(feature_path, file)
        data = np.load(filepath)

        # If file has explicit keys 'features' and 'labels'
        if "features" in data and "labels" in data:
            all_features.append(data["features"])
            all_labels.append(data["labels"])
        else:
            # Otherwise, guess keys (anything with 'feature' vs 'label')
            for key in data.files:
                if "feature" in key.lower():
                    all_features.append(data[key])
                elif "label" in key.lower():
                    all_labels.append(data[key])

X = np.concatenate(all_features, axis=0)
y = np.concatenate(all_labels, axis=0)

print(f"Loaded {len(all_features)} files → Features shape: {X.shape}, Labels shape: {y.shape}")

# Encode labels if categorical
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data Preprocessing Complete.\n")

# --- 2. Model Training and Hyperparameter Tuning ---
models_to_tune = {}
print("Step 2: Defining Models and Hyperparameter Search Spaces...")

# 2.1 Support Vector Machine (SVM)
svm_params = {'C': uniform(0.1, 10), 'gamma': uniform(0.001, 0.1), 'kernel': ['rbf']}
models_to_tune['SVM'] = (SVC(probability=True, random_state=42), svm_params)

# 2.2 Decision Tree
dt_params = {'max_depth': randint(5, 50), 'min_samples_split': randint(2, 20),
             'min_samples_leaf': randint(1, 20), 'criterion': ['gini', 'entropy']}
models_to_tune['Decision Tree'] = (DecisionTreeClassifier(random_state=42), dt_params)

# 2.3 Random Forest
rf_params = {'n_estimators': randint(100, 500), 'max_depth': randint(10, 100),
             'min_samples_split': randint(2, 20), 'min_samples_leaf': randint(1, 20)}
models_to_tune['Random Forest'] = (RandomForestClassifier(random_state=42, n_jobs=-1), rf_params)

# 2.4 AdaBoost
ada_params = {'n_estimators': randint(50, 500), 'learning_rate': uniform(0.01, 1.0)}
models_to_tune['AdaBoost'] = (AdaBoostClassifier(random_state=42), ada_params)

# 2.5 CatBoost
cat_params = {'iterations': randint(100, 500), 'learning_rate': uniform(0.01, 0.3),
              'depth': randint(4, 10), 'l2_leaf_reg': uniform(1, 10)}
models_to_tune['CatBoost'] = (CatBoostClassifier(random_state=42, silent=True), cat_params)

# 2.6 XGBoost
xgb_params = {'n_estimators': randint(100, 500), 'max_depth': randint(3, 10),
              'learning_rate': uniform(0.01, 0.3)}
models_to_tune['XGBoost'] = (XGBClassifier(eval_metric='mlogloss', random_state=42, use_label_encoder=False), xgb_params)

# 2.7 Gaussian Naive Bayes
gnb_params = {'var_smoothing': uniform(1e-10, 1e-7)}
models_to_tune['Gaussian NB'] = (GaussianNB(), gnb_params)

# 2.8 MLP Classifier
mlp_params = {'hidden_layer_sizes': [(50,), (100,), (50, 50)],
              'activation': ['tanh', 'relu'], 'alpha': uniform(0.0001, 0.01)}
models_to_tune['MLP Classifier'] = (MLPClassifier(random_state=42, max_iter=500), mlp_params)

# --- Tuning Loop ---
best_models = {}
for name, (model, params) in models_to_tune.items():
    print(f"Tuning {name}...")
    random_search = RandomizedSearchCV(model, param_distributions=params, n_iter=20, cv=3,
                                       random_state=42, n_jobs=-1)
    random_search.fit(X_train_scaled, y_train)
    best_models[name] = random_search.best_estimator_
    print(f"{name} tuning complete.")

# --- 3. Evaluation and Results ---
print("\nStep 3: Evaluating All Tuned Models...")
results = {}
for name, model in best_models.items():
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    results[(name, "Train")] = {
        'Accuracy': accuracy_score(y_train, y_train_pred),
        'Precision': precision_score(y_train, y_train_pred, average='macro'),
        'Recall': recall_score(y_train, y_train_pred, average='macro'),
        'F1-score': f1_score(y_train, y_train_pred, average='macro')
    }
    results[(name, "Test")] = {
        'Accuracy': accuracy_score(y_test, y_test_pred),
        'Precision': precision_score(y_test, y_test_pred, average='macro'),
        'Recall': recall_score(y_test, y_test_pred, average='macro'),
        'F1-score': f1_score(y_test, y_test_pred, average='macro')
    }

# --- 4. Final Report ---
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df.index = pd.MultiIndex.from_tuples(results_df.index, names=['Model', 'Dataset'])
print("\n--- COMPREHENSIVE PERFORMANCE REPORT ---")
print(results_df.round(3))


Step 1: Loading and Preprocessing Data...
Loaded 109 files → Features shape: (4898, 6), Labels shape: (4898,)
Data Preprocessing Complete.

Step 2: Defining Models and Hyperparameter Search Spaces...
Tuning SVM...
SVM tuning complete.
Tuning Decision Tree...
Decision Tree tuning complete.
Tuning Random Forest...
Random Forest tuning complete.
Tuning AdaBoost...
AdaBoost tuning complete.
Tuning CatBoost...
CatBoost tuning complete.
Tuning XGBoost...
XGBoost tuning complete.
Tuning Gaussian NB...
Gaussian NB tuning complete.
Tuning MLP Classifier...
MLP Classifier tuning complete.

Step 3: Evaluating All Tuned Models...

--- COMPREHENSIVE PERFORMANCE REPORT ---
                        Accuracy  Precision  Recall  F1-score
Model          Dataset                                       
SVM            Train       0.648      0.648   0.648     0.647
               Test        0.591      0.592   0.591     0.589
Decision Tree  Train       0.704      0.707   0.703     0.702
               Test   

In [3]:
pip install catboost


   ---------------------------------------- 0.0/102.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/102.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/102.4 MB ? eta -:--:--
   ---------------------------------------- 0.5/102.4 MB 1.2 MB/s eta 0:01:26
   ---------------------------------------- 0.8/102.4 MB 907.1 kB/s eta 0:01:53
   ---------------------------------------- 1.0/102.4 MB 1.0 MB/s eta 0:01:41
   ---------------------------------------- 1.0/102.4 MB 1.0 MB/s eta 0:01:41
    --------------------------------------- 1.6/102.4 MB 1.1 MB/s eta 0:01:32
    --------------------------------------- 1.8/102.4 MB 1.2 MB/s eta 0:01:26
    --------------------------------------- 1.8/102.4 MB 1.2 MB/s eta 0:01:26
    --------------------------------------- 2.4/102.4 MB 1.2 MB/s eta 0:01:23
   - -------------------------------------- 2.6/102.4 MB 1.2 MB/s eta 0:01:22
   - -------------------------------------- 2.9/102.4 MB 1.2 MB/s eta 0:01:25
   - 


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\samee\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip
